# 0. Dependencies

In [20]:
import torch
from torch.utils.data import Dataset, DataLoader

import torchvision
from torchvision.models.detection.rpn import AnchorGenerator
from torchvision.transforms import functional as F

import os, json, cv2, numpy as np, matplotlib.pyplot as plt

import albumentations as A # Library for augmentations

import time

import os
import sys
sys.path.append('pytorch_vision')

# from pytorch_vision import transforms, utils, engine, train
from utils import collate_fn
from engine import train_one_epoch, evaluate
from coco_utils import _coco_remove_images_without_annotations
import pycocotools

In [33]:
class ClassDataset(Dataset):

    def __init__(self, root, transform=None, demo=False):                
        self.root = root
        self.transform = transform
        self.demo = demo # Use demo=True if you need transformed and original images (for example, for visualization purposes)
        self.imgs_files = sorted(os.listdir(os.path.join(root, "images")))
        self.ids = list(range(len(self.imgs_files)))  # Add this line for compatibility with COCO-style utilities
        # self.annotations_files = sorted(os.listdir(os.path.join(root, "annotations")))
        # find the json annnotations file
        self.annotations_file = [f for f in os.listdir(os.path.join(root, "annotations")) if f.endswith('.json')][0] # Assuming there is only one json file in the annotations folder       
        self.root = root
        ann_path = os.path.join(self.root, "annotations", self.annotations_file)
        self.coco = pycocotools.coco.COCO(ann_path)
    
    def __getitem__(self, idx):
        import torch
        import cv2
        import json
        import numpy as np
        from torchvision.transforms import functional as F

        img_path = os.path.join(self.root, "images", self.imgs_files[idx])
        annotations_path = os.path.join(self.root, "annotations", self.annotations_file)

        img_original = cv2.imread(img_path)
        img_original = cv2.cvtColor(img_original, cv2.COLOR_BGR2RGB)

        with open(annotations_path) as f:
            data = json.load(f)
            data = data['annotations']
            objects = [obj for obj in data if obj['image_id'] == idx]
            bboxes_original = [obj['bbox'] for obj in objects]
            keypoints_original = [obj['keypoints'] for obj in objects]

            # Ensure keypoints shape: [num_instances, num_keypoints, 3]
            # If only one keypoint per object, this will be [N, 1, 3]
            keypoints_original = [
                [kp] if isinstance(kp[0], (int, float)) else kp
                for kp in keypoints_original
            ]

            # Filter out negative keypoints (optional, as in your code)
            if any(
                (kp[0][0] < 0 or kp[0][1] < 0)
                for kp in keypoints_original if len(kp) > 0
            ):
                next_idx = (idx + 1) % len(self)
                return self.__getitem__(next_idx)

            # Convert bboxes from [xmin, ymin, w, h] to [xmin, ymin, xmax, ymax]
            bboxes_original = [
                [bbox[0], bbox[1], bbox[0] + bbox[2], bbox[1] + bbox[3]]
                for bbox in bboxes_original
            ]

            labels = {0: 'angle_bar', 1: 'round_bar', 2: 'flat_bar'}
            bboxes_labels_original = [labels[obj['category_id']] for obj in objects]

        if self.transform:
            keypoints_original_flattened = [kp[0][:2] for kp in keypoints_original if len(kp) > 0]
            try:
                transformed = self.transform(
                    image=img_original,
                    bboxes=bboxes_original,
                    bboxes_labels=bboxes_labels_original,
                    keypoints=keypoints_original_flattened
                )
                img = transformed['image']
                bboxes = transformed['bboxes']
                img_h, img_w = img.shape[:2]
                keypoints = []
                for kp in transformed['keypoints']:
                    x, y = kp[0], kp[1]
                    v = 2
                    if not (0 <= x < img_w and 0 <= y < img_h):
                        x, y, v = 0, 0, 0
                    keypoints.append([[x, y, v]])
            except Exception as e:
                img = img_original
                bboxes = bboxes_original
                keypoints = [[[0, 0, 0]] for _ in keypoints_original]
        else:
            img, bboxes, keypoints = img_original, bboxes_original, keypoints_original

        # Convert to torch tensors
        bboxes = torch.as_tensor(bboxes, dtype=torch.float32)
        if bboxes.numel() == 0:
            bboxes = bboxes.new_zeros((0, 4))
        elif bboxes.ndim == 1:
            bboxes = bboxes.unsqueeze(0)

        num_keypoints = 1  # Set this to your actual number of keypoints per object
        keypoints = torch.as_tensor(keypoints, dtype=torch.float32)
        if keypoints.numel() == 0:
            keypoints = keypoints.new_zeros((0, num_keypoints, 3))
        target = {}
        target["boxes"] = bboxes
        labels_tensor = torch.as_tensor([1 for _ in range(bboxes.shape[0])], dtype=torch.int64)
        if labels_tensor.numel() == 0:
            labels_tensor = labels_tensor.new_zeros((0,), dtype=torch.int64)
        target["labels"] = labels_tensor
        target["keypoints"] = keypoints
        if bboxes.shape[0] == 0 or bboxes.shape[1] < 4:
            target["area"] = torch.as_tensor([], dtype=torch.float32)
        else:
            target["area"] = (bboxes[:, 3] - bboxes[:, 1]) * (bboxes[:, 2] - bboxes[:, 0])
        target["image_id"] = torch.tensor([idx])
        target["iscrowd"] = torch.zeros(bboxes.shape[0], dtype=torch.int64)
        img = F.to_tensor(img)

        # For original targets (for demo/visualization)
        bboxes_original = torch.as_tensor(bboxes_original, dtype=torch.float32)
        if bboxes_original.numel() == 0:
            bboxes_original = bboxes_original.new_zeros((0, 4))
        elif bboxes_original.ndim == 1:
            bboxes_original = bboxes_original.unsqueeze(0)
        target_original = {}
        target_original["boxes"] = bboxes_original
        labels_original = torch.as_tensor([1 for _ in range(bboxes_original.shape[0])], dtype=torch.int64)
        if labels_original.numel() == 0:
            labels_original = labels_original.new_zeros((0,), dtype=torch.int64)
        target_original["labels"] = labels_original
        keypoints_original_tensor = torch.as_tensor(keypoints_original, dtype=torch.float32)
        if keypoints_original_tensor.numel() == 0:
            keypoints_original_tensor = keypoints_original_tensor.new_zeros((0, num_keypoints, 3))
        target_original["keypoints"] = keypoints_original_tensor
        target_original["image_id"] = torch.tensor([idx])
        if bboxes_original.shape[0] == 0 or bboxes_original.shape[1] < 4:
            target_original["area"] = torch.as_tensor([], dtype=torch.float32)
        else:
            target_original["area"] = (bboxes_original[:, 3] - bboxes_original[:, 1]) * (bboxes_original[:, 2] - bboxes_original[:, 0])
        target_original["iscrowd"] = torch.zeros(bboxes_original.shape[0], dtype=torch.int64)
        img_original = F.to_tensor(img_original)

        if self.demo:
            return img, target, img_original, target_original
        else:
            return img, target
    
    def __len__(self):
        return len(self.imgs_files)

In [34]:
KEYPOINTS_FOLDER_TEST = '.\\datasets\\bars_keypoints_v3\\'
# KEYPOINTS_FOLDER_TEST = '.\\datasets\\bars_just_images\\'


In [35]:
dataset_test = ClassDataset(KEYPOINTS_FOLDER_TEST, transform=None, demo=False)


loading annotations into memory...
Done (t=0.06s)
creating index...
index created!


In [36]:
data_loader_test = DataLoader(dataset_test, batch_size=1, shuffle=False, collate_fn=collate_fn)

In [37]:
def get_model(num_keypoints, weights_path=None):
    
    anchor_generator = AnchorGenerator(sizes=(32, 64, 128, 256, 512), aspect_ratios=(0.25, 0.5, 0.75, 1.0, 2.0, 3.0, 4.0))
    model = torchvision.models.detection.keypointrcnn_resnet50_fpn(pretrained=False,
                                                                   pretrained_backbone=True,
                                                                   num_keypoints=num_keypoints,
                                                                   num_classes = 3, # Background is the first class, object is the second class
                                                                   rpn_anchor_generator=anchor_generator)

    if weights_path:
        state_dict = torch.load(weights_path)
        model.load_state_dict(state_dict)        
        
    return model

In [38]:
%%capture
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')

model = get_model(num_keypoints=1)
model.load_state_dict(torch.load('keypointsrcnn_weights_v3.pth', map_location=device))
model.to(device)
model.eval()

In [39]:
keypoints_classes_ids2names = {0: 'center'}

def visualize(image, bboxes, keypoints, image_original=None, bboxes_original=None, keypoints_original=None, fig_size=200):
    fontsize = 10

    for bbox in bboxes:
        start_point = (bbox[0], bbox[1])
        end_point = (bbox[2], bbox[3])
        image = cv2.rectangle(image.copy(), start_point, end_point, (0,255,0), 1)
    
    for kps in keypoints:
        for idx, kp in enumerate(kps):
            image = cv2.circle(image.copy(), tuple(kp), 2, (255,0,0), 10)
            # image = cv2.putText(image.copy(), " " + keypoints_classes_ids2names[idx], tuple(kp), cv2.FONT_HERSHEY_SIMPLEX, .5, (255,0,0), 3, cv2.LINE_AA)

    if image_original is None and keypoints_original is None:
        return image

    else:
        for bbox in bboxes_original:
            start_point = (bbox[0], bbox[1])
            end_point = (bbox[2], bbox[3])
            image_original = cv2.rectangle(image_original.copy(), start_point, end_point, (0,255,0), 1)
        
        for kps in keypoints_original:
            for idx, kp in enumerate(kps):
                image_original = cv2.circle(image_original, tuple(kp), 2, (255,0,0), 10)
                # image_original = cv2.putText(image_original, " " + keypoints_classes_ids2names[idx], tuple(kp), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (255,0,0), 3, cv2.LINE_AA)

        f, ax = plt.subplots(2,1, figsize=(fig_size, fig_size/2))

        return image_original
        

In [40]:
iterator = iter(data_loader_test)

# 1. Inference (rerun for next image)

Go to next batch and load images

In [41]:
for i in range(20):
    batch = next(iterator)


In [42]:
%%capture

if isinstance(batch, tuple) and len(batch) == 2:
    # Normal case: (images, targets)
    images, targets = batch
elif isinstance(batch, tuple) and len(batch) == 4:
    # Demo case: (images, targets, original_images, original_targets)
    images, targets, _, _ = batch
else:
    # Fallback
    print(f"Unexpected batch format: {type(batch)}, length: {len(batch) if isinstance(batch, tuple) else 'N/A'}")
    images, targets = batch[0], batch[1]

s_time = time.time()

with torch.no_grad():
    model.to(device)
    model.eval()
    output = model(images)

inf_time = time.time() - s_time

print("Predictions: \n", output)

In [43]:
image = (images[0].permute(1,2,0).detach().cpu().numpy() * 255).astype(np.uint8)

In [44]:
scores = output[0]['scores'].detach().cpu().numpy()

keypoints = []
bboxes = []

# Only process if we have any detections
if len(scores) > 0:
    high_scores_idxs = np.where(scores > 0.3)[0].tolist()  # Lower threshold for early training
    
    if len(high_scores_idxs) > 0:
        try:
            post_nms_idxs = torchvision.ops.nms(
                output[0]['boxes'][high_scores_idxs], 
                output[0]['scores'][high_scores_idxs], 
                0.3
            ).cpu().numpy()
            
            if len(post_nms_idxs) > 0:
                for kps in output[0]['keypoints'][high_scores_idxs][post_nms_idxs].detach().cpu().numpy():
                    keypoints.append([list(map(int, kp[:2])) for kp in kps])
                    
                for bbox in output[0]['boxes'][high_scores_idxs][post_nms_idxs].detach().cpu().numpy():
                    bboxes.append(list(map(int, bbox.tolist())))
        except Exception as e:
            print(f"Warning: Visualization error: {e}")

# block box plotting
bboxes = []

# 2. Run visualization

In [45]:
image = visualize(image, bboxes, keypoints)
print(f'Inference took {inf_time:.2f}s')


# cv2.imshow("Inference Result", image)
cv2.imwrite("inference_result.png", image)

Inference took 5.85s


True

In [ ]:
def load_images_from_directory(directory_path, device=None):
    """
    Load all images from a directory and prepare them for inference.
    
    Args:
        directory_path (str): Path to directory containing images
        device (torch.device): Device to load tensors to (if None, won't move to any device)
    
    Returns:
        list: List of image tensors ready for model inference
        list: List of original images (numpy arrays) for visualization
        list: List of image file names
    """
    if device is None:
        device = torch.device('cpu')
    
    # Get all image files
    valid_extensions = ['.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff']
    image_files = [f for f in os.listdir(directory_path) 
                   if any(f.lower().endswith(ext) for ext in valid_extensions)]
    
    if not image_files:
        raise ValueError(f"No image files found in {directory_path}")
    
    # Load and preprocess images
    images_tensor = []
    original_images = []
    
    for img_file in image_files:
        # Load image
        img_path = os.path.join(directory_path, img_file)
        img = cv2.imread(img_path)
        if img is None:
            print(f"Warning: Could not read image {img_path}")
            continue
        
        # Convert from BGR to RGB
        # img = cv2.cvtColor(img.copy(), cv2.COLOR_BGR2RGB)

        # # Convert back to uint8 numpy array for visualization
        # img_vis = (img * 255).astype(np.uint8)
        
        # # Invert colors if needed
        # img = 255 - img_vis
        
        # Store original image for visualization
        original_images.append(img)
        
        # Convert to tensor format required by the model
        img_tensor = F.to_tensor(img).to(device)
        images_tensor.append(img_tensor)


        
    return images_tensor, original_images, image_files

# Example usage:
def run_inference_on_directory(directory_path, model, device=None, score_threshold=0.3):
    """
    Run inference on all images in a directory and visualize results.
    
    Args:
        directory_path (str): Path to directory containing images
        model (torch.nn.Module): PyTorch model for inference
        device (torch.device): Device to run inference on
        score_threshold (float): Threshold for detection scores
    """
    if device is None:
        device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
    
    # Load images
    image_tensors, original_images, filenames = load_images_from_directory(directory_path, device)
    
    # Run inference
    model.to(device)
    model.eval()
    
    with torch.no_grad():
        for i, (img_tensor, orig_img, filename) in enumerate(zip(image_tensors, original_images, filenames)):
            # Measure inference time
            start_time = time.time()
            
            # Run inference (model expects a list of tensors)
            outputs = model([img_tensor])
            output = outputs[0]  # Get the first (and only) output
            
            inf_time = time.time() - start_time
            
            # Process results
            scores = output['scores'].detach().cpu().numpy()
            
            keypoints = []
            bboxes = []
            scores_ann = []
            
            # Process detections above threshold
            if len(scores) > 0:
                high_scores_idxs = np.where(scores > score_threshold)[0].tolist()
                
                if high_scores_idxs:
                    try:
                        post_nms_idxs = torchvision.ops.nms(
                            output['boxes'][high_scores_idxs], 
                            output['scores'][high_scores_idxs], 
                            0.3
                        ).cpu().numpy()
                        
                        if len(post_nms_idxs) > 0:
                            for kps in output['keypoints'][high_scores_idxs][post_nms_idxs].detach().cpu().numpy():
                                keypoints.append([list(map(int, kp[:2])) for kp in kps])
                                
                            for bbox in output['boxes'][high_scores_idxs][post_nms_idxs].detach().cpu().numpy():
                                bboxes.append(list(map(int, bbox.tolist())))

                            for score in output['scores'][high_scores_idxs][post_nms_idxs].detach().cpu().numpy():
                                scores_ann.append(list(map(int, score.tolist())))

                    except Exception as e:
                        print(f"Warning: Error processing detections: {e}")
            
            # Visualize
            # plt.figure(figsize=(10, 10))
            img_vis = visualize(orig_img, bboxes, keypoints, fig_size=10)
            # plt.title(f"File: {filename} - Inference time: {inf_time:.4f}s")
            cv2.imwrite(f".\\results_keypoints\\{filename}", img_vis
                        )
            print(f"Processed {filename} - Inference took {inf_time:.4f}s")
            print(f"Found {len(keypoints)} keypoints and {len(bboxes)} bounding boxes\n")


In [48]:
run_inference_on_directory('./test_images', model, device)

Processed 00004.png - Inference took 7.3070s
Found 30 keypoints and 30 bounding boxes

Processed 00104.png - Inference took 6.0588s
Found 32 keypoints and 32 bounding boxes

Processed 00143.png - Inference took 6.1312s
Found 30 keypoints and 30 bounding boxes

Processed 00164.png - Inference took 5.7422s
Found 0 keypoints and 0 bounding boxes

Processed 01.jpg - Inference took 5.9265s
Found 11 keypoints and 11 bounding boxes

Processed bus.jpg - Inference took 4.4518s
Found 0 keypoints and 0 bounding boxes

Processed dog.jpeg - Inference took 5.4156s
Found 12 keypoints and 12 bounding boxes

Processed IMG_9127.jpg - Inference took 7.3408s
Found 16 keypoints and 16 bounding boxes

Processed IMG_9130.jpg - Inference took 6.5486s
Found 16 keypoints and 16 bounding boxes

Processed IMG_9246.jpg - Inference took 5.7065s
Found 27 keypoints and 27 bounding boxes

Processed IMG_9266.jpg - Inference took 5.6992s
Found 26 keypoints and 26 bounding boxes

